In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Compute RI time series for case 0008-03, member 001, at 100 hPa.

1) 使用 EXTR 200 年 v'T' 4D，计算 100 hPa 上：
   - SIB 盒子 EHF 气候态和标准差（按 DOY）
   - CAN 盒子 EHF 气候态和标准差（按 DOY）

2) 使用 hindcast 0008-03 member1 的 VTH3d：
   - 取 100 hPa
   - 每一天算 SIB/CAN EHF
   - 用干净定义计算 RI(t) = z_sib(t) - z_can(t)

3) 输出：
   - RI_0008-03_member001_timeseries.csv
   - RI_0008-03_member001_timeseries.png
"""

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ================== 路径 ==================
ROOT_EHF = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"

EXTR_VTPRIME_4D = os.path.join(ROOT_EHF, "EHF_EXTR_vTprime_4D.nc")

HINDCAST_MEMBER1_0008_03 = (
    "/home/weiji/restart_exam/hindcast_data/0008-03/VTH3d/"
    "BWCN.e122.f19_g16.002.0008-03.001.cam.h3.VTH3d.nc"
)

OUT_CSV = os.path.join(ROOT_EHF, "RI_0008-03_member001_timeseries.csv")
OUT_PNG = os.path.join(ROOT_EHF, "RI_0008-03_member001_timeseries.png")

# ================== 盒子定义（SIB / CAN）==================
LAT_MIN = 45.0
LAT_MAX = 75.0

SIB_LON_MIN = 140.0
SIB_LON_MAX = 200.0

CAN_LON_MIN = 230.0   # 180E–270E
CAN_LON_MAX = 280.0


# ================== 小工具 ==================
def normalize_lon(da):
    """如果 lon 是 -180~180，就转成 0~360，并排序。"""
    if "lon" not in da.coords:
        return da
    lon = da["lon"].values
    if np.nanmin(lon) < 0:
        lon_new = (lon + 360.0) % 360.0
        da = da.assign_coords(lon=lon_new)
        da = da.sortby("lon")
    return da


def box_mean_coslat(da, lat_min, lat_max, lon_min, lon_max):
    """
    对 (time, lat, lon) 或 (lat, lon) 做 cos(lat) 加权 box 平均。
    """
    da = normalize_lon(da)
    sub = da.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    w = np.cos(np.deg2rad(sub["lat"]))
    return sub.weighted(w).mean(dim=("lat", "lon"))


def choose_first_var(ds):
    """简单暴力：拿第一个 data_var，当你没给 DataArray 命名时用。"""
    if len(ds.data_vars) == 0:
        raise ValueError("No data variables in dataset")
    vname = list(ds.data_vars)[0]
    return vname


# ==========================================================
# STEP 1: 从 EXTR v'T'（200 年）计算 100 hPa SIB/CAN 气候态和标准差
# ==========================================================
def compute_climatology_from_EXTR():
    print("\n=== STEP 1: 从 EXTR_vTprime_4D 计算 SIB/CAN 气候态 ===")
    print(f"[PATH] {EXTR_VTPRIME_4D}")

    ds_extr = xr.open_dataset(EXTR_VTPRIME_4D)
    print("[INFO] EXTR vTprime 4D dataset structure:")
    print(ds_extr)

    var_extr = choose_first_var(ds_extr)
    print(f"[INFO] 使用变量: {var_extr}")
    vt_extr = ds_extr[var_extr]     # (time, lev, lat, lon)

    # 垂直维（EXTR）：一般是 lev
    vert_dim = None
    for cand in ["lev", "plev", "plev_2", "level"]:
        if cand in vt_extr.dims:
            vert_dim = cand
            break

    if vert_dim is None:
        raise ValueError(f"找不到垂直维度，vt_extr.dims = {vt_extr.dims}")

    lev_vals = vt_extr[vert_dim].values
    lev_max = np.nanmax(lev_vals)
    print(f"[INFO] lev range: {lev_vals.min()} -> {lev_max}")

    # 简单判断单位是 hPa 还是 Pa
    if lev_max > 2000:   # 说明是 Pa
        target_level = 10000.0
    else:                # hPa
        target_level = 100.0

    vt_100 = vt_extr.sel({vert_dim: target_level}, method="nearest")
    print(f"[INFO] 选取 {vert_dim}={float(vt_100[vert_dim])} 这一层")
    print("[INFO] vt_100 dims:", vt_100.dims)

    time_extr = vt_100["time"]
    print(f"[INFO] EXTR time range: {time_extr.min().values} -> {time_extr.max().values}")
    print(f"[INFO] EXTR time length: {time_extr.size}")

    # Box 平均：SIB / CAN
    ehf_sib_ts = box_mean_coslat(vt_100, LAT_MIN, LAT_MAX, SIB_LON_MIN, SIB_LON_MAX)
    ehf_can_ts = box_mean_coslat(vt_100, LAT_MIN, LAT_MAX, CAN_LON_MIN, CAN_LON_MAX)

    print("\n[INFO] EHF SIB / CAN time series (EXTR):")
    print("  SIB:", ehf_sib_ts)
    print("  CAN:", ehf_can_ts)

    # DOY climatology
    sib_group = ehf_sib_ts.groupby("time.dayofyear")
    can_group = ehf_can_ts.groupby("time.dayofyear")

    ehf_sib_clim = sib_group.mean("time")
    ehf_can_clim = can_group.mean("time")

    ehf_sib_std = sib_group.std("time")
    ehf_can_std = can_group.std("time")

    print("\n[CLIM] SIB climatology DOY range:",
          int(ehf_sib_clim["dayofyear"].min()),
          "->",
          int(ehf_sib_clim["dayofyear"].max()))
    print("[CLIM] CAN climatology DOY range:",
          int(ehf_can_clim["dayofyear"].min()),
          "->",
          int(ehf_can_clim["dayofyear"].max()))

    # 检查 DOY=1 一天
    d1 = 1
    sib_clim_d1 = float(ehf_sib_clim.sel(dayofyear=d1))
    can_clim_d1 = float(ehf_can_clim.sel(dayofyear=d1))
    sib_std_d1  = float(ehf_sib_std.sel(dayofyear=d1))
    can_std_d1  = float(ehf_can_std.sel(dayofyear=d1))

    print(f"\n[CLIM] DOY=1 (Jan-01) climatology from EXTR:")
    print(f"  EHF_sib_clim(1) = {sib_clim_d1:.6g}")
    print(f"  EHF_can_clim(1) = {can_clim_d1:.6g}")
    print(f"  sigma_sib(1)    = {sib_std_d1:.6g}")
    print(f"  sigma_can(1)    = {can_std_d1:.6g}")

    ds_extr.close()

    return ehf_sib_clim, ehf_can_clim, ehf_sib_std, ehf_can_std


# ==========================================================
# STEP 2: 用 0008-03 member1 的 VTH3d 计算整段 RI(t)
# ==========================================================
def compute_RI_timeseries_0008_03(
    ehf_sib_clim, ehf_can_clim, ehf_sib_std, ehf_can_std
):
    print("\n=== STEP 2: 读取 0008-03 member1 VTH3d, 计算 RI(t) ===")
    print(f"[PATH] {HINDCAST_MEMBER1_0008_03}")

    ds_h = xr.open_dataset(HINDCAST_MEMBER1_0008_03)
    print("[INFO] Hindcast member1 dataset structure:")
    print(ds_h)

    # 找到变量名（一般是 VTH3d）
    var_h = None
    for cand in ["VTH3d", "EHF", "vT", "VT"]:
        if cand in ds_h.data_vars:
            var_h = cand
            break
    if var_h is None:
        raise KeyError(f"没有找到 VTH3d/EHF 类变量，vars={list(ds_h.data_vars)}")

    print(f"[INFO] 使用 hindcast 变量: {var_h}")
    vth = ds_h[var_h]   # dims: (time, plev_2, lat, lon) or similar

    # 垂直层
    vert_dim_h = None
    for cand in ["plev_2", "plev", "lev", "level"]:
        if cand in vth.dims:
            vert_dim_h = cand
            break

    if vert_dim_h is None:
        raise ValueError(f"hindcast 里找不到垂直维，dims={vth.dims}")

    lev_vals_h = vth[vert_dim_h].values
    lev_max_h = np.nanmax(lev_vals_h)
    if lev_max_h > 2000:
        target_level_h = 10000.0
    else:
        target_level_h = 100.0

    vth_100 = vth.sel({vert_dim_h: target_level_h}, method="nearest")
    print(f"[INFO] 选取 {vert_dim_h}={float(vth_100[vert_dim_h])} 这一层")
    print("[INFO] vth_100 dims:", vth_100.dims)

    time_h = vth_100["time"]
    print(f"[INFO] Hindcast time range: {time_h.min().values} -> {time_h.max().values}")
    print(f"[INFO] Hindcast time length: {time_h.size}")

    # box EHF time series
    ehf_sib_case = box_mean_coslat(vth_100, LAT_MIN, LAT_MAX, SIB_LON_MIN, SIB_LON_MAX)
    ehf_can_case = box_mean_coslat(vth_100, LAT_MIN, LAT_MAX, CAN_LON_MIN, CAN_LON_MAX)

    print("\n[INFO] 0008-03 member1 SIB/CAN EHF time series example:")
    print("  SIB:", ehf_sib_case.isel(time=0))
    print("  CAN:", ehf_can_case.isel(time=0))

    # 对齐 DOY，计算每一天的 z_sib, z_can, RI
    doy_case = time_h.dt.dayofyear
    z_sib = []
    z_can = []
    RI = []

    for i in range(time_h.size):
        d = int(doy_case.values[i])

        sib_clim_d = float(ehf_sib_clim.sel(dayofyear=d))
        can_clim_d = float(ehf_can_clim.sel(dayofyear=d))
        sib_std_d  = float(ehf_sib_std.sel(dayofyear=d))
        can_std_d  = float(ehf_can_std.sel(dayofyear=d))

        sib_case_d = float(ehf_sib_case.isel(time=i))
        can_case_d = float(ehf_can_case.isel(time=i))

        if sib_std_d != 0:
            z_sib_d = (sib_case_d - sib_clim_d) / sib_std_d
        else:
            z_sib_d = np.nan

        if can_std_d != 0:
            z_can_d = (can_case_d - can_clim_d) / can_std_d
        else:
            z_can_d = np.nan

        RI_d = z_sib_d - z_can_d

        z_sib.append(z_sib_d)
        z_can.append(z_can_d)
        RI.append(RI_d)

    z_sib = np.array(z_sib)
    z_can = np.array(z_can)
    RI = np.array(RI)

    ds_h.close()

    print("\n[RI] 0008-03 member1 RI time series summary:")
    print("  RI min =", np.nanmin(RI))
    print("  RI max =", np.nanmax(RI))
    print("  RI mean =", np.nanmean(RI))

    return time_h.values, z_sib, z_can, RI


# ==========================================================
# STEP 3: 画 RI–time 折线图 + 保存 CSV
# ==========================================================
def save_and_plot_RI(time_arr, RI):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    print("\n=== STEP 3: 保存 CSV 并绘制 RI 时间序列图 ===")

    # ---- 保存 CSV ----
    # time_arr 是 cftime.DatetimeNoLeap，先转成字符串存一下
    time_str = [str(t) for t in time_arr]

    df = pd.DataFrame({
        "time": time_str,
        "RI": RI,
    })
    df.to_csv(OUT_CSV, index=False)
    print(f"[OUT] RI time series saved to: {OUT_CSV}")

    # ---- 画图：用索引做 x 轴 ----
    x = np.arange(len(RI))

    plt.figure(figsize=(10, 4))
    plt.plot(x, RI, marker="o", linewidth=1)
    plt.axhline(0.0, linestyle="--")
    plt.xlabel("Time (index)")
    plt.ylabel("RI (z_sib - z_can)")
    plt.title("RI time series — case 0008-03, member001, 100 hPa")

    # 给一些稀疏的 tick，显示真正的日期字符串
    if len(x) > 0:
        step = max(1, len(x) // 8)   # 最多 8 个 tick
        xticks = x[::step]
        xticklabels = [time_str[i][:10] for i in xticks]  # 截到 YYYY-MM-DD
        plt.xticks(xticks, xticklabels, rotation=45, ha="right")

    plt.tight_layout()
    plt.savefig(OUT_PNG, dpi=150)
    plt.show()
    print(f"[OUT] RI time series figure saved to: {OUT_PNG}")



# ==========================================================
# Main
# ==========================================================
def main():
    ehf_sib_clim, ehf_can_clim, ehf_sib_std, ehf_can_std = compute_climatology_from_EXTR()
    time_arr, z_sib, z_can, RI = compute_RI_timeseries_0008_03(
        ehf_sib_clim, ehf_can_clim, ehf_sib_std, ehf_can_std
    )
    save_and_plot_RI(time_arr, RI)
    print("\n[Done] RI time series for 0008-03 member001 completed.")


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ================== 路径配置 ==================
ROOT_EHF = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"

# EXTR 200 年 v′T′ 4D，用来算气候态
EXTR_VTPRIME_4D = os.path.join(ROOT_EHF, "EHF_EXTR_vTprime_4D.nc")

# hindcast 根目录
VTH_ROOT = "/home/weiji/restart_exam/hindcast_data"

# 调试 case：0008-03
CASE = "0008-03"

OUT_CSV = os.path.join(ROOT_EHF, f"RI_U_corr_{CASE}_MarMay.csv")
OUT_PNG = os.path.join(ROOT_EHF, f"RI_U_corr_{CASE}_MarMay.png")

# ================== 盒子 & 物理配置 ==================
LAT_MIN, LAT_MAX = 45.0, 75.0

# ✅ 正确盒子
SIB_LON_MIN, SIB_LON_MAX = 140.0, 200.0   # Siberia: 45–75N, 140–200E
CAN_LON_MIN, CAN_LON_MAX = 230.0, 280.0   # Canada: 45–75N, 230–280E

TARGET_PLEV_HPA = 100.0

# RI 定义里仍然使用 15-day running mean
RM_WINDOW = 15

# ================== 小工具 ==================

def normalize_lon(da):
    """如果 lon 是 -180~180，就转成 0~360，并排序。"""
    if "lon" not in da.coords:
        return da
    lon = da["lon"].values
    if np.nanmin(lon) < 0:
        lon_new = (lon + 360.0) % 360.0
        da = da.assign_coords(lon=lon_new)
        da = da.sortby("lon")
    return da


def box_mean_coslat(da, lat_min, lat_max, lon_min, lon_max):
    """
    对 (time, lat, lon) 做 cos(lat) 加权 box 平均。
    """
    da = normalize_lon(da)
    sub = da.sel(lat=slice(lat_min, lat_max),
                 lon=slice(lon_min, lon_max))
    w = np.cos(np.deg2rad(sub["lat"]))
    return sub.weighted(w).mean(dim=("lat", "lon"))


def choose_first_var(ds):
    if len(ds.data_vars) == 0:
        raise ValueError("No data variables in dataset")
    return list(ds.data_vars)[0]


def select_level_plev_generic(ds, varname, target_hpa):
    """
    通用选层：压强坐标可能叫 plev / plev_2 / lev（Pa 或 hPa）。
    返回 DataArray(time, lat, lon)。
    """
    if isinstance(ds, xr.DataArray):
        ds = ds.to_dataset(name=varname)

    if "plev_2" in ds:
        pcoord = "plev_2"
    elif "plev" in ds:
        pcoord = "plev"
    elif "lev" in ds:
        pcoord = "lev"
    else:
        raise KeyError("No pressure coordinate (plev/plev_2/lev) in dataset.")

    lev_vals = ds[pcoord].values
    lev_max = np.nanmax(lev_vals)
    if lev_max > 2000:   # Pa
        plev_hpa = ds[pcoord] / 100.0
    else:                # hPa
        plev_hpa = ds[pcoord]

    idx = int(np.abs(plev_hpa - target_hpa).argmin())
    lev_val = float(plev_hpa.isel({pcoord: idx}).values)
    print(f"    [select_level] target={target_hpa} hPa, picked {pcoord}[{idx}]={lev_val:.1f} hPa")

    da = ds[varname].isel({pcoord: idx})
    return da

# ================== STEP 1: EXTR 气候态 ==================

def compute_climatology_from_EXTR():
    print("\n=== STEP 1: 从 EXTR_vTprime_4D 计算 SIB/CAN 气候态 ===")
    print(f"[PATH] {EXTR_VTPRIME_4D}")

    ds_extr = xr.open_dataset(EXTR_VTPRIME_4D)
    var_extr = choose_first_var(ds_extr)
    print(f"[INFO] 使用变量: {var_extr}")
    vt_extr = ds_extr[var_extr]

    vt_100 = select_level_plev_generic(ds_extr, var_extr, TARGET_PLEV_HPA)
    time_extr = vt_100["time"]
    print(f"[INFO] EXTR time range: {time_extr.min().values} -> {time_extr.max().values}")
    print(f"[INFO] EXTR length: {time_extr.size}")

    # SIB / CAN EHF 时间序列
    ehf_sib_ts = box_mean_coslat(vt_100, LAT_MIN, LAT_MAX, SIB_LON_MIN, SIB_LON_MAX)
    ehf_can_ts = box_mean_coslat(vt_100, LAT_MIN, LAT_MAX, CAN_LON_MIN, CAN_LON_MAX)

    sib_group = ehf_sib_ts.groupby("time.dayofyear")
    can_group = ehf_can_ts.groupby("time.dayofyear")

    ehf_sib_clim = sib_group.mean("time")
    ehf_can_clim = can_group.mean("time")
    ehf_sib_std  = sib_group.std("time")
    ehf_can_std  = can_group.std("time")

    ds_extr.close()
    return ehf_sib_clim, ehf_can_clim, ehf_sib_std, ehf_can_std

# ================== STEP 2: hindcast 上算 RI(time) ==================

def compute_member_RI(
    vth_file,
    ehf_sib_clim, ehf_can_clim,
    ehf_sib_std,  ehf_can_std,
):
    """
    对单个 hindcast VTH3d 文件：
    - 取 100 hPa
    - 限制 1–6 月
    - 算 SIB/CAN box mean EHF
    - 15-day rolling mean
    - 用 EXTR 气候态做 Z-score，得到 RI(time)
    """
    print(f"\n[RI] VTH file: {os.path.basename(vth_file)}")
    ds_h = xr.open_dataset(vth_file)

    var_h = None
    for cand in ["VTH3d", "EHF", "vT", "VT"]:
        if cand in ds_h.data_vars:
            var_h = cand
            break
    if var_h is None:
        raise KeyError(f"没有找到 VTH3d/EHF 类变量，data_vars={list(ds_h.data_vars)}")

    print(f"[INFO] 使用变量: {var_h}")
    vth_100 = select_level_plev_generic(ds_h, var_h, TARGET_PLEV_HPA)
    time_h = vth_100["time"]
    print(f"[INFO] hindcast time: {time_h.min().values} -> {time_h.max().values} "
          f"({time_h.size} steps)")

    # 只保留 1–6 月
    mon = time_h.dt.month
    mask = (mon >= 1) & (mon <= 6)
    vth_100 = vth_100.sel(time=mask)
    time_h = vth_100["time"]
    print(f"[INFO] Jan–Jun subset: {time_h.size} steps, "
          f"{time_h.values[0]} -> {time_h.values[-1]}")

    # EHF time series
    ehf_sib_case = box_mean_coslat(vth_100, LAT_MIN, LAT_MAX, SIB_LON_MIN, SIB_LON_MAX)
    ehf_can_case = box_mean_coslat(vth_100, LAT_MIN, LAT_MAX, CAN_LON_MIN, CAN_LON_MAX)

    # 15-day rolling mean
    ehf_sib_rm = ehf_sib_case.rolling(time=RM_WINDOW, center=True).mean()
    ehf_can_rm = ehf_can_case.rolling(time=RM_WINDOW, center=True).mean()
    valid = ehf_sib_rm.notnull() & ehf_can_rm.notnull()
    ehf_sib_rm = ehf_sib_rm.sel(time=valid)
    ehf_can_rm = ehf_can_rm.sel(time=valid)
    time_h = ehf_sib_rm["time"]

    if time_h.size == 0:
        ds_h.close()
        print("    WARNING: no valid time points after rolling.")
        return None

    doy = time_h.dt.dayofyear
    doy_da = xr.DataArray(doy.values, dims=["time"])

    sib_clim = ehf_sib_clim.sel(dayofyear=doy_da)
    can_clim = ehf_can_clim.sel(dayofyear=doy_da)
    sib_std  = ehf_sib_std.sel(dayofyear=doy_da)
    can_std  = ehf_can_std.sel(dayofyear=doy_da)

    sib_std = sib_std.where(sib_std != 0, np.nan)
    can_std = can_std.where(can_std != 0, np.nan)

    z_sib = (ehf_sib_rm - sib_clim) / sib_std
    z_can = (ehf_can_rm - can_clim) / can_std
    ri = z_sib - z_can   # (time,)

    ds_h.close()
    return ri


def load_U_100hPa_60N(vth_file, target_lat=60.0):
    """
    从对应的 U 文件读取 100 hPa、60°N 的 zonal-mean U(time)，1–6 月。
    """
    u_file = vth_file.replace("/VTH3d/", "/U/").replace(".VTH3d.nc", ".U.nc")
    if not os.path.exists(u_file):
        raise FileNotFoundError(f"U file not found: {u_file}")

    print(f"[U] U file: {os.path.basename(u_file)}")
    ds_u = xr.open_dataset(u_file)

    u_100 = select_level_plev_generic(ds_u, "U", TARGET_PLEV_HPA)
    time_u = u_100["time"]

    mon = time_u.dt.month
    mask = (mon >= 1) & (mon <= 6)
    u_100 = u_100.sel(time=mask)
    time_u = u_100["time"]
    print(f"[U] Jan–Jun subset: {time_u.size} steps, "
          f"{time_u.values[0]} -> {time_u.values[-1]}")

    # zonal mean
    u_zm = u_100.mean("lon")
    lat_idx = np.abs(u_zm["lat"] - target_lat).argmin().item()
    lat_val = float(u_zm["lat"].isel(lat=lat_idx).values)
    print(f"[U] nearest lat to {target_lat}°N: {lat_val:.2f}°N (index={lat_idx})")

    u_60N = u_zm.isel(lat=lat_idx)
    ds_u.close()
    return u_60N

# ================== STEP 3: 对 0008-03 全部 members 做相关性 ==================

# 1) 先算气候态
ehf_sib_clim, ehf_can_clim, ehf_sib_std, ehf_can_std = compute_climatology_from_EXTR()

# 2) 找这个 case 的所有 VTH3d members
vth_case_dir = os.path.join(VTH_ROOT, CASE, "VTH3d")
vth_files_all = sorted(glob.glob(os.path.join(vth_case_dir, "*.VTH3d.nc")))
if len(vth_files_all) == 0:
    raise RuntimeError(f"No VTH3d files found in {vth_case_dir}")

print(f"\n[MAIN] Case={CASE}, members found: {len(vth_files_all)}")

u_means = []
n_days_reflect = []
member_ids = []

for idx, vth_file in enumerate(vth_files_all, start=1):
    member_ids.append(idx)

    ri = compute_member_RI(
        vth_file,
        ehf_sib_clim, ehf_can_clim,
        ehf_sib_std,  ehf_can_std,
    )
    if ri is None:
        print(f"  [WARN] member {idx:03d}: RI is None, skip.")
        continue

    u_60N = load_U_100hPa_60N(vth_file)

    # 时间对齐（inner join），保证 RI 和 U 是同一批天
    ri_aligned, u_aligned = xr.align(ri, u_60N, join="inner")

    # 只取 3–5 月
    time = ri_aligned["time"]
    mon = time.dt.month
    mask_3_5 = (mon >= 3) & (mon <= 5)

    ri_3_5 = ri_aligned.sel(time=mask_3_5)
    u_3_5  = u_aligned.sel(time=mask_3_5)

    if ri_3_5.size == 0:
        print(f"  [WARN] member {idx:03d}: no data in Mar–May, skip.")
        continue

    # Y: 3–5 月 RI >= 1 的总天数（不再要求连续 10 天）
    n_reflect = int((ri_3_5 >= 1.0).sum().item())

    # X: 3–5 月 U(60N,100hPa) 的时间平均
    u_mean = float(u_3_5.mean().item())

    print(f"  [STAT] member {idx:03d}: U_mean(3–5)={u_mean:.2f} m/s, "
          f"days_RI>=1 (3–5)={n_reflect}")

    u_means.append(u_mean)
    n_days_reflect.append(n_reflect)

# 转成 numpy
u_means = np.array(u_means, dtype=float)
n_days_reflect = np.array(n_days_reflect, dtype=float)

# 只保留有效点
valid = np.isfinite(u_means) & np.isfinite(n_days_reflect)
u_means = u_means[valid]
n_days_reflect = n_days_reflect[valid]

# 计算相关系数
if u_means.size > 1:
    r = np.corrcoef(u_means, n_days_reflect)[0, 1]
else:
    r = np.nan
print(f"\n[RESULT] Pearson r (U_mean vs reflection days, Mar–May) = {r:.3f}")

# 保存 CSV
import pandas as pd
df_out = pd.DataFrame({
    "member": np.arange(1, len(member_ids)+1)[valid],
    "U_mean_3_5": u_means,
    "N_days_RI_ge_1_3_5": n_days_reflect,
})
df_out.to_csv(OUT_CSV, index=False)
print(f"[OUT] Saved stats to: {OUT_CSV}")

# 画散点图
plt.figure(figsize=(6, 5))
plt.scatter(u_means, n_days_reflect)
plt.xlabel("Mean U @100 hPa, 60°N (Mar–May) [m/s]")
plt.ylabel("Number of days with RI ≥ 1 (Mar–May)")
plt.title(f"{CASE}: RI–U relationship (r = {r:.2f})")
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=150)
plt.show()
print(f"[OUT] Figure saved to: {OUT_PNG}")
